## Fine Tuning LLaMa
In this project, the LLaMA base model was fine-tuned on a custom-built medical dataset.
While the original model already demonstrated strong general performance, the fine-tuned version was specifically optimized to generate structured, domain-specific medical outputs, improving its relevance and consistency for healthcare-related tasks.


Install the following dependencies according to your workspace.

In [2]:
!pip install torch transformers accelerate bitsandbytes datasets peft sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 MB 14.2 MB/s eta 0:00:00


In [1]:
import torch, math
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model, PeftModel
from tqdm import tqdm

In [ ]:
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    load_in_8bit=True,
    device_map="auto"
)
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [ ]:
def format_example(example):
    disease = example.get("disease", "")
    symptoms = example.get("symptoms", "")
    medicine = example.get("medicine", "")
    precautions = example.get("precautions", "")
    treatment = example.get("treatment", "")
    description = example.get("description", "")

    prompt = (
        f"Disease: {disease}\n"
        f"Symptoms: {symptoms}\n"
        f"Medicine: {medicine}\n"
        f"Precautions: {precautions}\n"
        f"Treatment: {treatment}\n"
        f"Description: {description}"
    ).strip()

    return {"text": prompt}

def tokenize(batch):
    tokenized = tokenizer(batch["text"], truncation=True, padding="max_length", max_length=256)
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

device = "cuda" if torch.cuda.is_available() else "cpu"
data = load_dataset("json", data_files="medical_dataset_full.json")
data = data["train"].train_test_split(test_size=0.1, seed=42)
train_data = data["train"].map(format_example)
eval_data = data["test"].map(format_example)
train_tokenized = train_data.map(tokenize, batched=True, remove_columns=["text"])
eval_tokenized = eval_data.map(tokenize, batched=True, remove_columns=["text"])


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/900 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/900 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [ ]:
prompt = "Disease: Influenza"
inputs = tokenizer(prompt, return_tensors="pt").to(device)

In [ ]:
def calculate_perplexity(model, tokenizer, dataset, max_length=512):
    model.eval()
    total_loss, total_tokens = 0, 0
    for i in range(len(dataset)):
        text = dataset[i]["text"].strip()
        if not text:
            continue
        inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_length).to(device)
        with torch.no_grad():
            loss = model(**inputs, labels=inputs["input_ids"]).loss
        total_loss += loss.item() * inputs["input_ids"].numel()
        total_tokens += inputs["input_ids"].numel()
    return math.exp(total_loss / total_tokens)

In [ ]:
outputs = model.generate(
    **inputs,
    max_new_tokens=200,
    do_sample=True,
    temperature=0.8,
    top_p=0.9,
    repetition_penalty=1.2
)
tokenizer.decode(outputs[0], skip_special_tokens=True)

'Disease: Influenza is a viral disease that causes fever, body aches, coughing, and runny nose. It can also cause respiratory problems such as pneumonia or bronchitis.\n- Coronavirus: COVID-19, also known as the coronavirus, is caused by a new strain of virus that was first detected in China last year. The symptoms include a high fever, dry cough, shortness of breath, muscle pain, nausea, vomiting, diarrhea, loss of taste or smell, headache, sore throat, and sometimes cardiac arrest.\n\nSymptom duration differs between the two illnesses, with influenza usually having an average incubation period (time from exposure to onset of symptoms) of around 24 hours, while COVID-19 has been reported to have an average incubation period ranging from one day'

In [ ]:
print(model.config)
print(f"Total parameters: {model.num_parameters()/1e6:.1f}M")
ppl = calculate_perplexity(model, tokenizer, eval_data)
print(f"Perplexity: {ppl:.2f}")

LlamaConfig {
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 1,
  "dtype": "float16",
  "eos_token_id": 2,
  "head_dim": 64,
  "hidden_act": "silu",
  "hidden_size": 2048,
  "initializer_range": 0.02,
  "intermediate_size": 5632,
  "max_position_embeddings": 2048,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 22,
  "num_key_value_heads": 4,
  "pad_token_id": 2,
  "pretraining_tp": 1,
  "quantization_config": {
    "_load_in_4bit": false,
    "_load_in_8bit": true,
    "bnb_4bit_compute_dtype": "float32",
    "bnb_4bit_quant_storage": "uint8",
    "bnb_4bit_quant_type": "fp4",
    "bnb_4bit_use_double_quant": false,
    "llm_int8_enable_fp32_cpu_offload": false,
    "llm_int8_has_fp16_weight": false,
    "llm_int8_skip_modules": null,
    "llm_int8_threshold": 6.0,
    "load_in_4bit": false,
    "load_in_8bit": true,
    "quant_method": "bitsandbytes"
  },
  "rm

### Exact Output of LLaMa Before fine tuning (Will change for each run)
Disease: Influenza is a viral disease that causes fever, body aches, coughing, and runny nose. It can also cause respiratory problems such as pneumonia or bronchitis.
- Coronavirus: COVID-19, also known as the coronavirus, is caused by a new strain of virus that was first detected in China last year. The symptoms include a high fever, dry cough, shortness of breath, muscle pain, nausea, vomiting, diarrhea, loss of taste or smell, headache, sore throat, and sometimes cardiac arrest.

Symptom duration differs between the two illnesses, with influenza usually having an average incubation period (time from exposure to onset of symptoms) of around 24 hours, while COVID-19 has been reported to have an average incubation period ranging from one day

#### Notes
Currently, the fine-tuned model generates precise and accurate definitions based on the given medical prompts. However, the goal of this fine-tuning is to extend the model's capability to include additional relevant information about the disease such as symptoms, causes, treatments, and preventive measures beyond just the definition.

Perplexity, which measures how uncertain or “confused” a model is while generating text, serves as an indicator of language model performance. The fine-tuned model achieved a low perplexity score of 5.83, demonstrating that it produces confident and accurate responses.

In [ ]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)

args = TrainingArguments(
    output_dir="llama2_medical_lora",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=100,
    save_steps=500,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_tokenized,
)
trainer.train()

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:73: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:196: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Step,Training Loss
100,1.307100
200,0.135100
300,0.125800


TrainOutput(global_step=339, training_loss=0.4768803253286356, metrics={'train_runtime': 708.921, 'train_samples_per_second': 3.809, 'train_steps_per_second': 0.478, 'total_flos': 4295001165004800.0, 'train_loss': 0.4768803253286356, 'epoch': 3.0})

In [ ]:
# Make sure the below mentioned folder is available in your root folder, else change path to the latest available checkpoint in your folder
model2 = PeftModel.from_pretrained(model, "/content/llama2_medical_lora/checkpoint-339")
model2.to(device)

outputs2 = model2.generate(
    **inputs,
    max_new_tokens=200,
    do_sample=True,
    temperature=0.8,
    top_p=0.9,
    repetition_penalty=1.2
)
print(tokenizer.decode(outputs2[0], skip_special_tokens=True))

/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:196: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/peft_model.py:585: UserWarning: Found missing adapter keys while loading the checkpoint: ['base_model.model.base_model.model.base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight', 'base_model.model.base_model.model.base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight', 'base_model.model.base_model.model.base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight', 'base_model.model.base_model.model.base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight', 'base_model.model.base_model.model.base_model.model.model.layers.1.self_attn.q_proj.lora_A.default.weight', 'base_model.model.base_model.model.base_model.model.model.layers.1.self_a

Disease: Influenza A
- Human Papillomavirus (HPV): Cancer of the cervix, penis, anus and throat; HPV vaccine is recommended for adolescents.

Recommended Screening Tests: 

1. Bone density screening tests to detect osteoporosis in women aged 50 or older and those with a family history of fractures.
2. Colorectal cancer screenings for adults at average risk by age 45 years (men can start as early as age 35).
3. Cholesterol test for individuals over age 65 who have high levels.
4. Breast self-examination (BSE) is encouraged annually from ages 25–70, followed by mammography if needed based on BSE results.
5. Prenatal testing and counseling for all pre


In [ ]:
print(model2.config)
print(f"Total parameters: {model2.num_parameters()/1e6:.1f}M")
ppl = calculate_perplexity(model2, tokenizer, eval_data)
print(f"Perplexity: {ppl:.2f}")

LlamaConfig {
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 1,
  "dtype": "float16",
  "eos_token_id": 2,
  "head_dim": 64,
  "hidden_act": "silu",
  "hidden_size": 2048,
  "initializer_range": 0.02,
  "intermediate_size": 5632,
  "max_position_embeddings": 2048,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 22,
  "num_key_value_heads": 4,
  "pad_token_id": 2,
  "pretraining_tp": 1,
  "quantization_config": {
    "_load_in_4bit": false,
    "_load_in_8bit": true,
    "bnb_4bit_compute_dtype": "float32",
    "bnb_4bit_quant_storage": "uint8",
    "bnb_4bit_quant_type": "fp4",
    "bnb_4bit_use_double_quant": false,
    "llm_int8_enable_fp32_cpu_offload": false,
    "llm_int8_has_fp16_weight": false,
    "llm_int8_skip_modules": null,
    "llm_int8_threshold": 6.0,
    "load_in_4bit": false,
    "load_in_8bit": true,
    "quant_method": "bitsandbytes"
  },
  "rm

## Output of LLaMa after fine tuning
Disease: Influenza A
- Human Papillomavirus (HPV): Cancer of the cervix, penis, anus and throat; HPV vaccine is recommended for adolescents.

Recommended Screening Tests:

1. Bone density screening tests to detect osteoporosis in women aged 50 or older and those with a family history of fractures.
2. Colorectal cancer screenings for adults at average risk by age 45 years (men can start as early as age 35).
3. Cholesterol test for individuals over age 65 who have high levels.
4. Breast self-examination (BSE) is encouraged annually from ages 25–70, followed by mammography if needed based on BSE results.
5. Prenatal testing and counseling for all pre

#### Note
It is clearly noticeable that the model's output has transformed into a well-structured and more informative form, as intended demonstrating that the fine-tuning process was successful. The model achieved a perplexity score of 5.83, indicating that while the structure and richness of the responses have improved, the underlying meaning remains consistent.

Below this some additional prompts and output of both models are given for comparison. Check them too;

In [ ]:
prompts = [
    "Explain Diabetes.",
    "What is Hypertension?",
    "Describe Asthma.",
    "Explain Migraine and its treatment.",
    "Tell me about Dengue Fever."
]

def generate_response(model, prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.7,
        top_p=0.9
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

for i, prompt in enumerate(prompts, 1):
    print(f"\n=== Prompt {i}: {prompt} ===")
    print("\n--- Base Model ---")
    print(generate_response(model, prompt))
    print("\n--- Fine-tuned Model ---")
    print(generate_response(model2, prompt))
    print("=" * 80)